In [ ]:
"""
MODEL SCRIPT (ASSUMES CLEANING IS DONE)

Goal:
- Build a robust valuation model using:
  1) Transfermarkt (TM) valuation labels + contract context
  2) EA (SoFIFA/FC) attributes + EA "value" (kept in dataset)
  3) FBref performance features (your Fbref_Final_Data is a 23/24 + 24/25 average)

Key design choices:
- Target = blended market value (TM + EA), blended in log-space for stability
- Features include EA skills/traits + FBref performance + TM contract/age context
- IMPORTANT: If EA "value" is included in the blended target, it must NOT be used as a feature,
  otherwise the model will partially learn the target directly (leakage).
- Contract years issue: if you already fixed it, we still implement a safe policy:
  set implausibly large contract_years_left to NaN and add a 'contract_years_stale' flag.

Ensemble:
- StackingRegressor with heterogeneous base models:
  - HistGradientBoostingRegressor (strong on tabular)
  - ExtraTreesRegressor (diverse tree-bagging)
  - RandomForestRegressor (stable baseline)
  - Ridge (linear anchor)
  Meta-model: Ridge

Evaluation:
- Time split by TM valuation date (because valuations are time-indexed)
- Since FBref is a 23/24+24/25 average, restrict training to 2023+ valuations
  (or change to your preferred window if you accept the temporal assumption).
"""

In [ ]:
%pip install scikit-learn pandas numpy matplotlib seaborn

import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import StackingRegressor

In [ ]:
TM_PATH = "../transfermarkt_merged_players_with_valuation.csv"
EA_PATH = "../player_stats_cleaned.csv"
FBREF_PATH = "../Fbref_Final_Data.csv"

In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def clean_name(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("ascii")
    x = re.sub(r"[^a-z\s\-']", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def safe_to_datetime(s):
    return pd.to_datetime(s, errors="coerce")

In [ ]:
# -----------------------------
# Load cleaned datasets
# -----------------------------
tm = pd.read_csv(TM_PATH)
ea = pd.read_csv(EA_PATH)
fb = pd.read_csv(FBREF_PATH)

# Parse dates
tm["date"] = safe_to_datetime(tm.get("date"))
tm["date_of_birth"] = safe_to_datetime(tm.get("date_of_birth"))
tm["contract_expiration_date"] = safe_to_datetime(tm.get("contract_expiration_date"))
ea["dob"] = safe_to_datetime(ea.get("dob"))

# Join keys
tm["clean_name"] = tm["name"].map(clean_name)
ea["clean_name"] = ea["name"].map(clean_name)
fb["clean_name"] = fb["Player"].map(clean_name)

# Coerce values
tm["tm_value"] = pd.to_numeric(tm.get("market_value_in_eur_valuation"), errors="coerce")
ea["ea_value"] = pd.to_numeric(ea.get("value"), errors="coerce")

# Drop obviously unusable rows
tm = tm.dropna(subset=["clean_name", "date", "date_of_birth", "tm_value"])
ea = ea.dropna(subset=["clean_name", "dob", "ea_value"])


In [ ]:
# -----------------------------
# Merge: TM ↔ EA on (clean_name + DOB)
# -----------------------------
df = tm.merge(
    ea,
    left_on=["clean_name", "date_of_birth"],
    right_on=["clean_name", "dob"],
    how="inner",
    suffixes=("_tm", "_ea"),
)

# Merge: add FBref features by clean_name (FBref is your averaged performance table)
fb_feat = fb.drop(columns=[c for c in ["Unnamed: 0", "Player"] if c in fb.columns], errors="ignore")
df = df.merge(fb_feat, on="clean_name", how="left")


In [ ]:
# -----------------------------
# Target: blended value in log-space
# -----------------------------
df = df.dropna(subset=["tm_value", "ea_value"])
df["target_value_blend"] = 0.5 * (np.log1p(df["tm_value"]) + np.log1p(df["ea_value"]))

# IMPORTANT:
# If EA value is part of the blended target, DO NOT use EA "value" as a feature.
# You said you'd like to keep EA game values in general, but "value" must be excluded from X
# when it contributes to y.
# (You can still keep EA wage/release_clause etc.)
# If you want to include EA "value" as a feature, then do NOT blend it into the target.


# -----------------------------
# Temporal restriction (recommended given FBref averaging window)
# -----------------------------
# Your FBref dataset summarizes 23/24 + 24/25. To avoid training on 2012 labels with 2024 performance,
# restrict to valuations from 2023 onward. Adjust if your project design differs.
df = df[df["date"].dt.year >= 2023].copy()

# If dataset becomes too small, you can relax to 2022+ etc, but understand the temporal assumption.


# -----------------------------